In [1]:
import sys

print(sys.executable)

C:\Users\jugal\langchain-env\Scripts\python.exe


# Wikipedia Retrievers

In [2]:
from langchain_community.retrievers import WikipediaRetriever

In [3]:
# Initialize wikipedia retriever.

wiki_retriever = WikipediaRetriever(
    top_k_results = 2,
    language = 'en')

# Define a query
query = "Indian independance and freedom fighters of India."

# Get relevant Wiki Documents
docs = wiki_retriever.invoke(query)

In [4]:
print(docs)

[Document(metadata={'title': 'Sardar (1993 film)', 'summary': "Sardar is a 1993 Indian biographical drama film on Sardar Vallabhbhai Patel, one of India's greatest freedom fighters, directed by Ketan Mehta and written by noted playwright Vijay Tendulkar and Hriday Lani. The film was screened retrospective on 12 August 2016 at the Independence Day Film Festival jointly presented by the Indian Directorate of Film Festivals and Ministry of Defence, commemorating 70th Indian Independence Day.", 'source': 'https://en.wikipedia.org/wiki/Sardar_(1993_film)'}, page_content="Sardar is a 1993 Indian biographical drama film on Sardar Vallabhbhai Patel, one of India's greatest freedom fighters, directed by Ketan Mehta and written by noted playwright Vijay Tendulkar and Hriday Lani. The film was screened retrospective on 12 August 2016 at the Independence Day Film Festival jointly presented by the Indian Directorate of Film Festivals and Ministry of Defence, commemorating 70th Indian Independence D

In [5]:
for i, doc in enumerate(docs):
    print(f"---Result {i+1}---")
    print(f'''Content:
    {doc.page_content}''')

    print()
    print()


---Result 1---
Content:
    Sardar is a 1993 Indian biographical drama film on Sardar Vallabhbhai Patel, one of India's greatest freedom fighters, directed by Ketan Mehta and written by noted playwright Vijay Tendulkar and Hriday Lani. The film was screened retrospective on 12 August 2016 at the Independence Day Film Festival jointly presented by the Indian Directorate of Film Festivals and Ministry of Defence, commemorating 70th Indian Independence Day.


== Plot ==
The film begins with a young Sardar Patel playing cards with his friends and ridiculing Mahatma Gandhi and his policies to achieve independence. His views change however, when he is introduced to Gandhi by his brother, and upon listening to a lecture delivered by Gandhi, he joins him in his struggle. Sardar then successfully organises various Satyagrahas throughout Gujarat. The film then moves to the age of the Quit India Movement and India's freedom. Sardar is instrumental in convincing the working committee of the INC an

# 2) Vector Store Retriever

In [6]:
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

from dotenv import load_dotenv
from langchain_chroma import Chroma

In [7]:
load_dotenv()

True

In [8]:
# Creating document objects

doc1 = Document(page_content = """Avenged Lala Lajpat Rai’s death by killing Saunders and
shook the British by bombing the Central Assembly.""",
               metadata = {'freedom_fighter': 'Bhagat Singh'},
               id = '1')


doc2 = Document(page_content = '''Organized youth through the Hindustan Socialist Republican 
Association (HSRA) and played a key role in revolutionary activities against the British.''',
               metadata = {'freedom_fighter': 'Sukhdev'},
               id = '2')


doc3 = Document(page_content = '''Actively participated in the plan to kill Saunders and fought
as a fearless revolutionary alongside Bhagat Singh and Sukhdev.''',
               metadata = {'freedom_fighter': 'Rajguru'},
               id = '3')


doc4 = Document(page_content = '''Led HSRA after the death of Ram Prasad Bismil, carried out armed 
resistance, and fought till his last breath to avoid British capture.''',
               metadata = {'freedom_fighter': 'Chandra Shekhar Azad'},
               id = '4')


doc5 = Document(page_content = '''Fired the first shot of revolt in 1857 against British officers,
sparking India’s first large-scale armed uprising for independence.''',
               metadata = {'freedom_fighter': 'Mangal Pandey'},
               id = '5')


doc6 = Document(page_content = '''Avenged the Jallianwala Bagh massacre by assassinating Michael O’Dwyer in London.''',
               metadata = {'freedom_fighter': 'Udham Singh'},
               id = '6')

In [9]:
docs = [doc1, doc2, doc3, doc4, doc5, doc6]

In [10]:
print(doc5)

page_content='Fired the first shot of revolt in 1857 against British officers,
sparking India’s first large-scale armed uprising for independence.' metadata={'freedom_fighter': 'Mangal Pandey'}


In [11]:
# Initializing Vector Store

vec_store = Chroma.from_documents(
    documents = docs,
    embedding = OpenAIEmbeddings(),
    collection_name = "India",
    persist_directory = "freedom_fighters_db"
    
)

In [39]:
# Converting the above vector_store to a retriever

retriever = vec_store.as_retriever(search_type = 'similarity',
                                   search_kwargs = {'k': 2})    # k = 2 means we want 2 results in return of our query

# We can also use 'mmr' as search type

In [14]:
# Searching

result = retriever.invoke("Who took the revenge of Jaliyaanwala Baugh masscre?")

In [18]:
result

[Document(id='6', metadata={'freedom_fighter': 'Udham Singh'}, page_content='Avenged the Jallianwala Bagh massacre by assassinating Michael O’Dwyer in London.'),
 Document(id='1', metadata={'freedom_fighter': 'Bhagat Singh'}, page_content='Avenged Lala Lajpat Rai’s death by killing Saunders and\nshook the British by bombing the Central Assembly.')]

In [27]:
for i, doc in enumerate(result):
    print(f"---Result {i+1}---")
    print(f'''Name: {doc.metadata['freedom_fighter']}
Content: {doc.page_content}''')

    print()
    print()


---Result 1---
Name: Udham Singh
Content: Avenged the Jallianwala Bagh massacre by assassinating Michael O’Dwyer in London.


---Result 2---
Name: Bhagat Singh
Content: Avenged Lala Lajpat Rai’s death by killing Saunders and
shook the British by bombing the Central Assembly.




# Doubt:
We just searched a relevant document by converting vector_store into a retriever, but in our last session, we did this same using `similarity_search_with_score()` method. Then what is the difference in both?

The difference is, `similarity_search_with_score()` uses same method of searching the relevant document but if we convert vector_store to a retriever, we can use any method to search the relevant document. We can define our custom function or use some predefined methods to for searching. `We will learn this later`

`Note`: Here, method means way/ proccess of searching

# Now, Go to word file and see MMR